# Section 4: Self-Service Analytics via Cortex Analyst Demo
**ACME IM | Distribution Insights**

Demonstrates the Semantic View + Cortex Analyst REST API.
Natural language → verified SQL → results. No SQL knowledge required.

In [ ]:
import os, tomllib, json, urllib.request, urllib.error
from cryptography.hazmat.primitives.serialization import load_pem_private_key
from snowflake.snowpark import Session
from snowflake.cortex import Complete
import snowflake.connector
import pandas as pd

# Load keypair auth from connections.toml (works locally; get_active_session() used in SiS)
_CONN_NAME = os.environ.get('SNOWFLAKE_DEFAULT_CONNECTION_NAME', 'your_connection')
with open(os.path.expanduser('~/.snowflake/connections.toml'), 'rb') as f:
    _cfg = tomllib.load(f)[_CONN_NAME]

_key_path = os.path.expanduser(_cfg['private_key_path'])
with open(_key_path, 'rb') as f:
    _private_key = load_pem_private_key(f.read(), password=None)

_params = {k: v for k, v in _cfg.items() if k != 'private_key_path'}
_params['private_key'] = _private_key

# Snowpark Session
session = Session.builder.configs(_params).create()
session.sql('USE WAREHOUSE INGEST').collect()
session.sql('USE DATABASE ANALYTICS_DEV_DB').collect()
session.sql('USE SCHEMA ANALYTICS_DEV_DB.DISTRIBUTION').collect()

# REST token for Cortex Analyst
_SF_HOST = f"{_cfg['account']}.snowflakecomputing.com"
_raw_conn = snowflake.connector.connect(**_params)
_SF_TOKEN = _raw_conn.rest.token

print(f'Connected: {session.get_current_account()}')
print(f'Context: {session.get_current_database()}.{session.get_current_schema()}')


In [ ]:
# Cortex Analyst REST API helper
def ask_analyst(question, sv='ANALYTICS_DEV_DB.DISTRIBUTION.DISTRIBUTION_INSIGHTS_SV'):
    url = f'https://{_SF_HOST}/api/v2/cortex/analyst/message'
    payload = {
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'semantic_view': sv,
    }
    req = urllib.request.Request(
        url, data=json.dumps(payload).encode(),
        headers={'Authorization': f'Snowflake Token="{_SF_TOKEN}"',
                 'Content-Type': 'application/json'},
        method='POST'
    )
    try:
        with urllib.request.urlopen(req) as r:
            return json.loads(r.read())
    except urllib.error.HTTPError as e:
        return {'error': json.loads(e.read())}


# Demo: wholesaler questions answered with verified SQL
questions = [
    'Who are the top 5 advisors by AUM?',
    'Which advisors are at risk of attrition by territory?',
    'Show me the open pipeline value by territory',
    'Give me a summary of our Platinum tier advisors',
]

for q in questions:
    print(f'\nQ: {q}')
    resp = ask_analyst(q)
    if 'error' in resp:
        print(f'  ERROR: {resp["error"]}')
        continue
    for msg in resp.get('message', {}).get('content', []):
        if msg['type'] == 'text':
            print(f'  {msg["text"][:200]}')
        elif msg['type'] == 'sql':
            vqr = msg.get('confidence', {}).get('verified_query_used')
            label = f'[VQR:{vqr["name"]}]' if vqr else '[AI-GEN]'
            print(f'  SQL {label}: {msg["statement"][:250]}...')
